In [ ]:
!pip install autogluon

In [ ]:
import pandas as pd
from datetime import datetime, timedelta, timezone
import requests
from bs4 import BeautifulSoup

def fetch_brightsky_data(start_date: datetime, end_date: datetime, station_id: str) -> pd.DataFrame | None:
    TARGET_TIMEZONE = 'Europe/Berlin'
    start_utc = start_date.astimezone(timezone.utc) if start_date.tzinfo else start_date.replace(tzinfo=timezone.utc)
    end_utc = end_date.astimezone(timezone.utc) if end_date.tzinfo else end_date.replace(tzinfo=timezone.utc)
    start_str = start_utc.isoformat(timespec='seconds')
    end_str = end_utc.isoformat(timespec='seconds')
    params = {'dwd_station_id': station_id, 'date': start_str, 'last_date': end_str}

    print(f"Lade Wetterdaten von Bright Sky für den Zeitraum (in UTC): {start_str} bis {end_str}...")
    try:
        response = requests.get("https://api.brightsky.dev/weather", params=params, timeout=30)
        response.raise_for_status()
        data = response.json().get('weather', [])
        if not data:
            print("Keine Wetterdaten für den angefragten Zeitraum gefunden.")
            return pd.DataFrame()
        df = pd.DataFrame(data)
        print(f"Erfolgreich {len(df)} stündliche Wetter-Datenpunkte geladen.")
        return df
    except requests.exceptions.RequestException as e:
        print(f"Netzwerk- oder API-Fehler beim Abrufen der Wetterdaten: {e}")
        return None

def get_prepared_weather_data():
    TARGET_TIMEZONE = 'Europe/Berlin'
    TAGE_VERGANGENHEIT = 40
    TAGE_ZUKUNFT = 8
    now_local = datetime.now().astimezone()
    start_date = now_local - timedelta(days=TAGE_VERGANGENHEIT)
    end_date = now_local + timedelta(days=TAGE_ZUKUNFT)

    df_raw = fetch_brightsky_data(start_date, end_date, "03379")
    if df_raw is None or df_raw.empty:
        print("Download der Wetterdaten fehlgeschlagen. Überspringe Wetter-Integration.")
        return pd.DataFrame()

    print(f"\nVerarbeite Wetterdaten und konvertiere zu Zeitzone '{TARGET_TIMEZONE}'...")
    wetter_df = df_raw[['timestamp', 'temperature', 'precipitation', 'pressure_msl']].copy()
    wetter_df['timestamp'] = pd.to_datetime(wetter_df['timestamp'])
    wetter_df.set_index('timestamp', inplace=True)
    
    # Bright Sky liefert UTC, hier konvertieren wir in die Zielzone
    wetter_df.index = wetter_df.index.tz_convert(TARGET_TIMEZONE)
    wetter_df.sort_index(inplace=True)
    wetter_df = wetter_df[~wetter_df.index.duplicated(keep='first')]
    wetter_df.rename(columns={'temperature': 'lufttemperatur_c', 'precipitation': 'niederschlag_mm', 'pressure_msl': 'pressure'}, inplace=True)

    wetter_df['niederschlag_mm'] = wetter_df['niederschlag_mm'].fillna(0)
    wetter_df['lufttemperatur_c'] = wetter_df['lufttemperatur_c'].interpolate(method='time')
    wetter_df['pressure'] = wetter_df['pressure'].interpolate(method='time')

    print("Resample Wetterdaten auf 1-Stunden-Intervall...")
    wetter_1h = wetter_df.resample('1h').agg({
        'lufttemperatur_c': 'mean',
        'niederschlag_mm': 'sum',
        'pressure': 'mean'
    }).round(2)
    return wetter_1h

def fetch_data_from_url(url, column_name):
    print(f"-> Processing URL for: {column_name}")
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, timeout=20, headers=headers)
        response.raise_for_status()
        html_content = response.content.decode('utf-8')
    except Exception as e:
        print(f"Fehler beim Laden der URL: {e}")
        return pd.DataFrame()
    
    soup = BeautifulSoup(html_content, 'html.parser')
    table = soup.find("table", class_="tblsort") or soup.find("table", class_="datentabelle")
    if not table: return pd.DataFrame()
    
    headers = [h.get_text(strip=True) for h in table.find('thead').find_all("th")]
    df_headers = headers if any('Uhrzeit' in s for s in headers) else ['Datum/Uhrzeit'] + headers[1:]
    rows = table.find('tbody').find_all("tr")
    data = []
    for row in rows:
        cells = row.find_all(["td", "th"])
        data.append({df_headers[i]: cell.get_text(strip=True) for i, cell in enumerate(cells) if i < len(df_headers)})
    
    df = pd.DataFrame(data)
    if 'Datum/Uhrzeit' in df.columns:
        df['timestamp'] = pd.to_datetime(df['Datum/Uhrzeit'], format='%d.%m.%Y %H:%M', errors='coerce')
    elif 'Datum' in df.columns and 'Uhrzeit' in df.columns:
        df['timestamp'] = pd.to_datetime(df['Datum'] + ' ' + df['Uhrzeit'], format="%d.%m.%Y %H:%M", errors='coerce')
    
    df.dropna(subset=['timestamp'], inplace=True)
    
    # Spalte für Messwert finden
    target_col = [c for c in df.columns if column_name.split('_')[0].lower() in c.lower()][0]
    df_final = df[["timestamp", target_col]].copy()
    df_final.rename(columns={target_col: column_name}, inplace=True)
    df_final[column_name] = pd.to_numeric(df_final[column_name].astype(str).str.replace(",", "."), errors='coerce')
    
    return df_final

# --- Main execution ---
end_date = datetime.now()
start_date = end_date - timedelta(days=40)
wassertemperatur_url = f"https://www.gkd.bayern.de/de/fluesse/wassertemperatur/bayern/muenchen-himmelreichbruecke-16515005/messwerte/tabelle?beginn={start_date.strftime('%d.%m.%Y')}&ende={end_date.strftime('%d.%m.%Y')}"

# 1. Daten laden
df_wt = fetch_data_from_url(wassertemperatur_url, "wassertemp")

# 2. KEY FIX: Zeitumstellung robust handhaben (Frühling & Herbst)
# 'nonexistent' fängt den 29.03.2026 02:00 Uhr ab, 'ambiguous' den Herbst.
df_wt['timestamp'] = df_wt['timestamp'].dt.tz_localize(
    'Europe/Berlin', 
    ambiguous='infer', 
    nonexistent='shift_forward'
)

# 3. Resampling erst NACH der Lokalisierung
df_wt = df_wt.set_index('timestamp').resample('1h').first().reset_index()

# 4. Wetterdaten holen
df_wetter = get_prepared_weather_data()

# 5. Mergen (beide sind nun Europe/Berlin aware)
df_merged = pd.merge(
    df_wt, 
    df_wetter.reset_index().rename(columns={'lufttemperatur_c': 'airtemp'}), 
    on='timestamp', 
    how='outer'
)

df_merged.set_index('timestamp', inplace=True)
df_merged = df_merged[df_merged.index.notna()].sort_index().interpolate(method='time').ffill().bfill()

# 6. Feature Shifting (Vorschau-Werte)
df_merged['airtemp_96'] = df_merged['airtemp'].shift(-96)
df_merged['pressure_96'] = df_merged['pressure'].shift(-96)
df_merged.drop(columns=['airtemp', 'pressure'], inplace=True)

# 7. Finalisierung: Zurück zu UTC für konsistente Speicherung/Verarbeitung
df_merged.index = df_merged.index.tz_convert('UTC')

# 8. Melting für Output
df_long = pd.melt(df_merged.reset_index(), id_vars=['timestamp'], value_vars=['wassertemp', 'airtemp_96', 'pressure_96'])
df_long.columns = ['date', 'cols', 'data']
df_long['cols'] = pd.Categorical(df_long['cols'], categories=['wassertemp', 'airtemp_96', 'pressure_96'], ordered=True)
df_long = df_long.sort_values(by=['cols', 'date'])

display(df_long.head())

In [ ]:
!git clone https://github.com/obwohl/ts_proba_cuda
%cd ts_proba_cuda
!git checkout a8de694266a629124687a8f2b9fcfdba15a3590c


In [ ]:
%cd ..
df_long.to_csv('ts_proba_cuda/df_long.csv', index=False)
!pip install optuna

In [ ]:
!python /kaggle/working/ts_proba_cuda/run_single_forecast.py \
    --checkpoint /kaggle/working/ts_proba_cuda/checkpoints/best_model.pt \
    --data-file /kaggle/working/ts_proba_cuda/df_long.csv \
    --output-csv /kaggle/working/ts_proba_cuda/inference.csv




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

# Ensure the 'date' column is in datetime format before we begin
df_long['date'] = pd.to_datetime(df_long['date'])
last_timestamp = df_long['date'].max()

# --- Run All Backtests ---

print("--- Starting Backtests ---")

# --- Prepare and run -96h backtest ---
print("\n[1/3] Preparing and running -96h backtest...")
backtest_96_end_date = last_timestamp - pd.Timedelta(hours=96)
df_long_backtest_96_corrected = df_long[df_long['date'] <= backtest_96_end_date]
df_long_backtest_96_corrected.to_csv('ts_proba_cuda/df_long_backtest_96_corrected.csv', index=False)
!python /kaggle/working/ts_proba_cuda/run_single_forecast.py \
    --checkpoint /kaggle/working/ts_proba_cuda/checkpoints/best_model.pt \
    --data-file /kaggle/working/ts_proba_cuda/df_long_backtest_96_corrected.csv \
    --output-csv /kaggle/working/ts_proba_cuda/inference_backtest_96_corrected.csv

# --- Prepare and run -192h backtest ---
print("\n[2/3] Preparing and running -192h backtest...")
backtest_192_end_date = last_timestamp - pd.Timedelta(hours=192)
df_long_backtest_192_corrected = df_long[df_long['date'] <= backtest_192_end_date]
df_long_backtest_192_corrected.to_csv('ts_proba_cuda/df_long_backtest_192_corrected.csv', index=False)
!python /kaggle/working/ts_proba_cuda/run_single_forecast.py \
    --checkpoint /kaggle/working/ts_proba_cuda/checkpoints/best_model.pt \
    --data-file /kaggle/working/ts_proba_cuda/df_long_backtest_192_corrected.csv \
    --output-csv /kaggle/working/ts_proba_cuda/inference_backtest_192_corrected.csv

# --- Prepare and run -288h backtest ---
print("\n[3/3] Preparing and running -288h backtest...")
backtest_288_end_date = last_timestamp - pd.Timedelta(hours=288)
df_long_backtest_288_corrected = df_long[df_long['date'] <= backtest_288_end_date]
df_long_backtest_288_corrected.to_csv('ts_proba_cuda/df_long_backtest_288_corrected.csv', index=False)
!python /kaggle/working/ts_proba_cuda/run_single_forecast.py \
    --checkpoint /kaggle/working/ts_proba_cuda/checkpoints/best_model.pt \
    --data-file /kaggle/working/ts_proba_cuda/df_long_backtest_288_corrected.csv \
    --output-csv /kaggle/working/ts_proba_cuda/inference_backtest_288_corrected.csv

print("\n--- Backtests Complete. Loading data for plotting. ---")


# --- Load All Forecast Data ---
df_inference = pd.read_csv('/kaggle/working/ts_proba_cuda/inference.csv', parse_dates=[0], index_col=0)
df_inference_backtest_96_corr = pd.read_csv('/kaggle/working/ts_proba_cuda/inference_backtest_96_corrected.csv', parse_dates=[0], index_col=0)
df_inference_backtest_192_corr = pd.read_csv('/kaggle/working/ts_proba_cuda/inference_backtest_192_corrected.csv', parse_dates=[0], index_col=0)
df_inference_backtest_288_corr = pd.read_csv('/kaggle/working/ts_proba_cuda/inference_backtest_288_corrected.csv', parse_dates=[0], index_col=0)


In [ ]:
# --- Final Plotting ---

# Define the style primer
primer = {
  "theme_color": "#231F20",
  "style": {
    "lines.linewidth": 1.0, "lines.linestyle": "-", "font.family": "sans-serif",
    "font.size": 10, "text.color": "#231F20", "axes.facecolor": "#FFFFFF",
    "axes.edgecolor": "#231F20", "axes.linewidth": 0.8, "axes.grid": True,
    "axes.labelsize": 10, "axes.labelweight": "normal", "axes.labelcolor": "#231F20",
    "axes.prop_cycle": cycler(color=["#1771F1", "#F85C50", "#35D073", "#FFC11E", "#8E44AD"]),
    "xtick.major.size": 2, "xtick.minor.size": 1, "xtick.major.width": 0.8,
    "xtick.minor.width": 0.6, "xtick.major.top": True, "xtick.major.bottom": True,
    "xtick.minor.top": True, "xtick.minor.bottom": True, "xtick.color": "#231F20", "xtick.labelsize": 8,
    "ytick.major.size": 2, "ytick.minor.size": 1, "ytick.major.width": 0.8,
    "ytick.minor.width": 0.6, "ytick.color": "#231F20", "ytick.major.left": True,
    "ytick.major.right": True, "ytick.minor.left": True, "ytick.minor.right": True,
    "grid.color": "#231F20", "grid.linestyle": ":", "grid.linewidth": 0.4,
    "grid.alpha": 1.0, "legend.frameon": False, "legend.edgecolor": "#231F20",
    "figure.figsize": [15, 9], "figure.dpi": 96, "figure.facecolor": "#FFFFFF",
    "figure.edgecolor": "#FFFFFF"
  }
}
plt.rcParams.update(primer['style'])

# Define plotting parameters
channel = 'wassertemp'
fig, ax = plt.subplots()

colors = primer['style']['axes.prop_cycle'].by_key()['color']
quantile_pairs = [(0.01, 0.99), (0.05, 0.95), (0.25, 0.75)]
alphas = [0.1, 0.15, 0.2]
quantile_labels = ['q0.01-q0.99', 'q0.05-q0.95', 'q0.25-q0.75']

# Plot Historical Data for wassertemp on the main axis
historical_data = df_long[df_long['cols'] == channel]
ax.plot(historical_data['date'], historical_data['data'], label='Historical Wassertemp', color='black', linestyle='--')

# Helper function to plot a forecast
def plot_forecast(df_forecast, label, color):
    median_col = f"{channel}_q0.5"
    ax.plot(df_forecast.index, df_forecast[median_col], label=label, color=color)
    for j, (q_low, q_high) in enumerate(quantile_pairs):
        col_low = f"{channel}_q{q_low}"
        col_high = f"{channel}_q{q_high}"
        if col_low in df_forecast.columns and col_high in df_forecast.columns:
            ax.fill_between(df_forecast.index, df_forecast[col_low], df_forecast[col_high],
                            alpha=alphas[j], color=color, lw=0)


# Plot all four forecasts for wassertemp
plot_forecast(df_inference, 'Forecast Wassertemp', colors[0])
plot_forecast(df_inference_backtest_96_corr, 'Backtest -96h', colors[1])
plot_forecast(df_inference_backtest_192_corr, 'Backtest -192h', colors[2])
plot_forecast(df_inference_backtest_288_corr, 'Backtest -288h', colors[3])

# Plot the unshifted air temperature forecast on the same axis
ax.plot(df_wetter.index, df_wetter['lufttemperatur_c'], label='Air Temp (DWD)', color='purple', linestyle=':', linewidth=1.5, alpha=0.6)


# Add invisible artists for the legend
for j, label in enumerate(quantile_labels):
    ax.fill_between([], [], [], color='gray', alpha=alphas[j], label=label)

# Final plot adjustments
plot_start_date = df_inference_backtest_288_corr.index.min()
plot_end_date = df_inference.index.max() # Set the end date to the last point of the main forecast
ax.set_xlim(left=plot_start_date, right=plot_end_date) # Set both start and end limits


# --- CHIRURGISCHER EINGRIFF -BLOCK START ---
# Wir holen uns die Wetter-Daten, die WIRKLICH im aktuellen Zoom-Fenster liegen
visible_wetter = df_wetter.loc[(df_wetter.index >= plot_start_date) & (df_wetter.index <= plot_end_date), 'lufttemperatur_c']

# Wir berechnen das Min/Max aus sichtbarem Wetter UND der Vorhersage (damit nichts abgeschnitten wird)
# Hinweis: Wir nehmen q0.01 und q0.99 der Vorhersage für die Sicherheit
y_view_min = min(visible_wetter.min(), df_inference[f"{channel}_q0.01"].min())
y_view_max = max(visible_wetter.max(), df_inference[f"{channel}_q0.99"].max())

# Wir setzen die Y-Achse neu mit einem kleinen Puffer (z.B. 1 Grad oben/unten)
ax.set_ylim(y_view_min - 0.5, y_view_max + 0.5)
# --- CHIRURGISCHER EINFÜGE-BLOCK ENDE ---

# Main Y-axis
ax.set_title(f'Eisbach - backtesting and forecast')
ax.set_xlabel('Date')
ax.set_ylabel('Temperatur (°C)')


# Combine legends
lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, loc='upper left')

# --- Save the figure ---
# Save as a high-resolution PNG file
file_path = '/kaggle/working/eisbach_new.png'
plt.savefig(file_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor(), edgecolor='none')
print(f"Plot saved to: {file_path}")

plt.show()

In [ ]:
from datetime import datetime, timedelta, timezone
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import os
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.timeseries.splitter import ExpandingWindowSplitter
from autogluon.common import space
from contextlib import redirect_stdout
from scipy.stats import randint, uniform
from bokeh.plotting import figure, show
from bokeh.palettes import Category10

from bokeh.models import HoverTool, ColumnDataSource, Legend, Segment, Text, VBar, LinearAxis, Range1d
from bokeh.layouts import column

import logging
from collections import deque
import time
from bokeh.io import save
import re
import boto3
from botocore.exceptions import NoCredentialsError, ClientError
from kaggle_secrets import UserSecretsClient

def fetch_brightsky_data(start_date: datetime, end_date: datetime, station_id: str) -> pd.DataFrame | None:
    TARGET_TIMEZONE = 'Europe/Berlin'
    start_utc = start_date.astimezone(timezone.utc) if start_date.tzinfo else start_date.replace(tzinfo=timezone.utc)
    end_utc = end_date.astimezone(timezone.utc) if end_date.tzinfo else end_date.replace(tzinfo=timezone.utc)
    start_str = start_utc.isoformat(timespec='seconds')
    end_str = end_utc.isoformat(timespec='seconds')
    params = {'dwd_station_id': station_id, 'date': start_str, 'last_date': end_str}

    print(f"Lade Wetterdaten von Bright Sky für den Zeitraum (in UTC): {start_str} bis {end_str}...")
    try:
        response = requests.get("https://api.brightsky.dev/weather", params=params, timeout=30)
        response.raise_for_status()
        data = response.json().get('weather', [])
        if not data:
            print("Keine Wetterdaten für den angefragten Zeitraum gefunden.")
            return pd.DataFrame()
        df = pd.DataFrame(data)
        return df
    except requests.exceptions.RequestException as e:
        print(f"Netzwerk- oder API-Fehler beim Abrufen der Wetterdaten: {e}")
        return None

def get_prepared_weather_data():
    TARGET_TIMEZONE = 'Europe/Berlin'
    TAGE_VERGANGENHEIT = 370
    TAGE_ZUKUNFT = 8
    now_local = datetime.now().astimezone()
    start_date = now_local - timedelta(days=TAGE_VERGANGENHEIT)
    end_date = now_local + timedelta(days=TAGE_ZUKUNFT)

    df_raw = fetch_brightsky_data(start_date, end_date, "03379")
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()

    wetter_df = df_raw[['timestamp', 'temperature', 'precipitation']].copy()
    wetter_df['timestamp'] = pd.to_datetime(wetter_df['timestamp'])
    wetter_df.set_index('timestamp', inplace=True)
    wetter_df.index = wetter_df.index.tz_convert(TARGET_TIMEZONE)
    wetter_df.sort_index(inplace=True)
    wetter_df = wetter_df[~wetter_df.index.duplicated(keep='first')]
    wetter_df.rename(columns={'temperature': 'lufttemperatur_c', 'precipitation': 'niederschlag_mm'}, inplace=True)
    wetter_df['niederschlag_mm'] = wetter_df['niederschlag_mm'].fillna(0)
    wetter_df['lufttemperatur_c'] = wetter_df['lufttemperatur_c'].interpolate(method='time')

    wetter_1h = wetter_df.resample('1h').agg({'lufttemperatur_c': 'mean', 'niederschlag_mm': 'sum'}).round(2)
    return wetter_1h

def fetch_data_from_url(url, column_name):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, timeout=20, headers=headers)
        response.raise_for_status()
        html_content = response.content.decode('utf-8')
    except Exception: return pd.DataFrame()
    soup = BeautifulSoup(html_content, 'html.parser')
    table = soup.find("table", class_="tblsort") or soup.find("table", class_="datentabelle")
    if not table: return pd.DataFrame()
    headers = [h.get_text(strip=True) for h in table.find('thead').find_all("th")]
    df_headers = headers if any('Uhrzeit' in s for s in headers) else ['Datum/Uhrzeit'] + headers[1:]
    rows = table.find('tbody').find_all("tr")
    data = []
    for row in rows: 
        cells = row.find_all(["td", "th"])
        data.append({df_headers[i]: cell.get_text(strip=True) for i, cell in enumerate(cells) if i < len(df_headers)})
    df = pd.DataFrame(data)
    if 'Datum/Uhrzeit' in df.columns:
        df['timestamp'] = pd.to_datetime(df['Datum/Uhrzeit'], format='%d.%m.%Y %H:%M', errors='coerce')
    elif 'Datum' in df.columns and 'Uhrzeit' in df.columns:
        df['timestamp'] = pd.to_datetime(df['Datum'] + ' ' + df['Uhrzeit'], format="%d.%m.%Y %H:%M", errors='coerce')
    df.dropna(subset=['timestamp'], inplace=True)
    target_header = column_name.split('_')[0]
    df_final = df[["timestamp", target_header]].copy()
    df_final.rename(columns={target_header: column_name}, inplace=True)
    df_final[column_name] = pd.to_numeric(df_final[column_name].astype(str).str.replace(",", "."), errors='coerce')
    return df_final

def main():
    wetter_data_1h = get_prepared_weather_data()
    end_date = datetime.now() - timedelta(hours=1)
    start_date = end_date - timedelta(days=364)
    
    urls_and_columns = {f"https://www.gkd.bayern.de/de/fluesse/wassertemperatur/bayern/muenchen-himmelreichbruecke-16515005/messwerte/tabelle?beginn={start_date.strftime('%d.%m.%Y')}&ende={end_date.strftime('%d.%m.%Y')}": "Wassertemperatur [°C]_München Himmelreichbruecke"}
    all_dfs = [fetch_data_from_url(url, col) for url, col in urls_and_columns.items()]
    merged_data = all_dfs[0].sort_values('timestamp')
    
    merged_data.rename(columns={merged_data.columns[1]: "wassertemp"}, inplace=True)
    water_hourly = merged_data.set_index("timestamp").resample("1h").median().reset_index()
    water_hourly["item_id"] = "eisbach_temp"
    
    data = TimeSeriesDataFrame.from_data_frame(water_hourly)
    PREDICTION_LENGTH = 64
    predictor = TimeSeriesPredictor(prediction_length=PREDICTION_LENGTH, target="wassertemp", eval_metric="SQL", verbosity=0)
    predictor.fit(data, hyperparameters={"Chronos": [{"model_path": "bolt_base", "ag_args": {"name_suffix": "bolt_base"}}]}, time_limit=300)
    
    future_pred = predictor.predict(data)
    preds_for_csv = future_pred.copy().reset_index(level='item_id', drop=True)

    # FIXED: Handhabung von Duplikaten vor dem Merge
    if not wetter_data_1h.empty:
        # Sicherstellen, dass der Index keine Duplikate hat
        preds_for_csv = preds_for_csv[~preds_for_csv.index.duplicated(keep='first')]
        wetter_data_1h = wetter_data_1h[~wetter_data_1h.index.duplicated(keep='first')]
        
        # Zeitzonen-Angleichung
        if preds_for_csv.index.tz is None:
            preds_for_csv.index = preds_for_csv.index.tz_localize('Europe/Berlin', ambiguous='infer', nonexistent='shift_forward')
        
        preds_for_csv = preds_for_csv.merge(wetter_data_1h, left_index=True, right_index=True, how='left')

    preds_for_csv[["0.1", "0.5", "0.9"]].to_csv("eisbach_predictions.csv", float_format="%.1f")
    print("Erfolgreich gespeichert.")

if __name__ == "__main__":
    main()

In [ ]:
!rm -rf /kaggle/working/ts_proba_cuda